# ML-08 — Model Build: Refresh Opportunity Scoring

This notebook moves from the Week-4 rule baseline to learned models for the Refresh / Content Opportunity Scoring lane. The evaluation uses the same anonymized starter data, a client-aware holdout, and ranking metrics so the comparison is decision-useful rather than complexity-driven.

## 1. Method choice and why

The lane asks **which content should be reviewed first**, so the model should produce a ranking score rather than only a hard yes/no decision. I use `is_declining_label` as the observed target and compare three readable classifiers: Logistic Regression, a shallow Decision Tree, and Random Forest. Their positive-class probabilities become ranking scores.

Logistic Regression is the simplest interpretable starting point. The Decision Tree tests whether a small amount of nonlinear structure helps. Random Forest is included as a stronger but still controlled model. I select the model using Precision@50 because the operational question is which small set should be reviewed first. I do not use `trend_pct` or `trend_direction` as features because they define the target. IDs are used only for the client holdout.

In [7]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, average_precision_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

RANDOM_STATE = 42
DATA_URL = 'https://raw.githubusercontent.com/Roselyn-Koech/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv'
local_candidates = [Path('data/raw/content_refresh_anonymized.csv'), Path('/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv')]
DATA_PATH = next((p for p in local_candidates if p.exists()), None)
df = pd.read_csv(DATA_PATH) if DATA_PATH else pd.read_csv(DATA_URL)

# Match the repository preparation rules used before modeling.
df = df[(df['impressions_90d'] > 0) & (df['content_age_days'] >= 90)].copy()
df = df.drop_duplicates('content_id').reset_index(drop=True)
df['is_declining_label'] = df['trend_direction'].astype(str).str.lower().eq('down').astype(int)
for c in ['search_volume','competition','cpc','word_count','char_count','impressions_90d','clicks_90d','pageviews_90d','sessions_90d','users_90d','engaged_sessions_90d','ai_sessions_90d','scroll_events_90d','days_with_impressions','days_with_sessions','content_age_days','days_since_last_update','ctr','avg_position','engagement_rate','scroll_rate','ai_traffic_pct']:
    df[c] = pd.to_numeric(df[c], errors='coerce').replace([np.inf, -np.inf], np.nan).fillna(0)
df['log_impressions_90d'] = np.log1p(df['impressions_90d'])
df['log_clicks_90d'] = np.log1p(df['clicks_90d'])
df['log_sessions_90d'] = np.log1p(df['sessions_90d'])
df['log_ai_sessions_90d'] = np.log1p(df['ai_sessions_90d'])

numeric_features = ['search_volume','competition','cpc','word_count','char_count','log_impressions_90d','log_clicks_90d','log_sessions_90d','log_ai_sessions_90d','days_with_impressions','days_with_sessions','content_age_days','days_since_last_update','ctr','avg_position','engagement_rate','scroll_rate','ai_traffic_pct']
categorical_features = ['competition_level','content_type','main_intent','age_tier','freshness_tier','word_count_tier','impression_tier','position_tier']
for c in categorical_features:
    df[c] = df[c].fillna('unknown').astype(str)

print(f'Rows prepared: {len(df):,}')
print(f'Columns available: {len(df.columns)}')
print(f'Declining label rate: {df.is_declining_label.mean():.3f}')

Rows prepared: 30,000
Columns available: 49
Declining label rate: 0.542


## 2. Split design

A client holdout is the preferred split because rows from the same client can share content and measurement patterns. Holding out whole clients is more conservative than randomly splitting rows and avoids letting the model learn client-specific patterns from both sides. With 32 clients, 20% gives a small but usable group holdout. The random seed is fixed at 42.

In [8]:
all_idx = np.arange(len(df))
clients = df['client_id'].fillna('unknown').astype(str)
unique_clients = clients.drop_duplicates().to_numpy()
rng = np.random.default_rng(RANDOM_STATE)
shuffled_clients = rng.permutation(unique_clients)
test_client_count = max(1, int(round(len(shuffled_clients) * 0.20)))
test_clients = set(shuffled_clients[:test_client_count])
test_mask = clients.isin(test_clients).to_numpy()
train_idx = all_idx[~test_mask]
test_idx = all_idx[test_mask]

# Safety check: if the grouped split produces a one-class test set, fall back to a stratified row split.
if df.iloc[train_idx]['is_declining_label'].nunique() < 2 or df.iloc[test_idx]['is_declining_label'].nunique() < 2:
    train_idx, test_idx = train_test_split(all_idx, test_size=0.20, random_state=RANDOM_STATE, stratify=df['is_declining_label'])
    split_strategy = 'stratified_row_holdout'
else:
    split_strategy = 'client_holdout'

print('Split strategy:', split_strategy)
print('Train rows:', len(train_idx))
print('Test rows:', len(test_idx))
print('Held-out clients:', sorted(test_clients))

Split strategy: client_holdout
Train rows: 27675
Test rows: 2325
Held-out clients: ['client_0b918943df', 'client_1a6562590e', 'client_4fc82b26ae', 'client_98a3ab7c34', 'client_d4735e3a26', 'client_f74efabef1']


## 3. Train + compare vs my baseline

The baseline is rebuilt inside this notebook so it is evaluated on exactly the same test rows as the learned models. It follows the Week-4 rule: visibility, freshness risk, position opportunity, and depth gap. The comparison uses ROC AUC, average precision, and especially Precision@20 and Precision@50.

In [9]:
def percentile_rank(s):
    return pd.to_numeric(s, errors='coerce').fillna(0).rank(method='average', pct=True).fillna(0)


def minmax(s):
    x = pd.to_numeric(s, errors='coerce').replace([np.inf, -np.inf], np.nan).fillna(0)
    lo, hi = x.min(), x.max()
    return (
        pd.Series(np.zeros(len(x)), index=x.index)
        if hi == lo
        else (x - lo) / (hi - lo)
    )


# -----------------------------
# Week 4 baseline
# -----------------------------

df['visibility_score'] = percentile_rank(
    np.log1p(df['impressions_90d'])
)

df['freshness_risk_score'] = percentile_rank(
    df['days_since_last_update']
)

df['position_opportunity_score'] = (
    (1 - minmax(df['avg_position'].clip(lower=1, upper=50)))
    * df['visibility_score']
    * (df['avg_position'] > 0).astype(int)
)

df['depth_gap_score'] = (
    (1 - percentile_rank(df['word_count']))
    * df['visibility_score']
)

df['baseline_refresh_score'] = (
    0.40 * df['visibility_score']
    + 0.30 * df['freshness_risk_score']
    + 0.25 * df['position_opportunity_score']
    + 0.05 * df['depth_gap_score']
).clip(0, 1)


# -----------------------------
# Model definitions
# -----------------------------

preprocessor = ColumnTransformer([
    (
        'num',
        StandardScaler(),
        numeric_features
    ),
    (
        'cat',
        OneHotEncoder(handle_unknown='ignore'),
        categorical_features
    ),
])


models = {
    'logistic_regression': Pipeline([
        (
            'prep',
            preprocessor
        ),
        (
            'model',
            LogisticRegression(
                class_weight='balanced',
                max_iter=1000,
                random_state=RANDOM_STATE
            )
        )
    ]),

    'decision_tree': Pipeline([
        (
            'prep',
            preprocessor
        ),
        (
            'model',
            DecisionTreeClassifier(
                class_weight='balanced',
                max_depth=5,
                min_samples_leaf=50,
                random_state=RANDOM_STATE
            )
        )
    ]),

    'random_forest': Pipeline([
        (
            'prep',
            preprocessor
        ),
        (
            'model',
            RandomForestClassifier(
                class_weight='balanced_subsample',
                max_depth=10,
                min_samples_leaf=25,
                n_estimators=200,
                n_jobs=-1,
                random_state=RANDOM_STATE
            )
        )
    ])
}


# -----------------------------
# Features and target
# -----------------------------

X = df[numeric_features + categorical_features]

y = df['is_declining_label'].astype(int)


# -----------------------------
# Precision@K
# -----------------------------

def precision_at_k(y_true, scores, k):
    order = np.argsort(np.asarray(scores))[::-1][:min(k, len(scores))]
    return float(np.asarray(y_true)[order].mean())


# -----------------------------
# Evaluation metrics
# -----------------------------

def metrics(y_true, scores):

    pred = (np.asarray(scores) >= 0.5).astype(int)

    return {
        'ROC AUC': roc_auc_score(y_true, scores),
        'Avg precision': average_precision_score(y_true, scores),
        'Precision@20': precision_at_k(y_true, scores, 20),
        'Precision@50': precision_at_k(y_true, scores, 50),
        'Recall': recall_score(
            y_true,
            pred,
            zero_division=0
        ),
        'F1': f1_score(
            y_true,
            pred,
            zero_division=0
        )
    }


# -----------------------------
# Evaluate Week 4 baseline
# -----------------------------

baseline_scores = (
    df.iloc[test_idx]['baseline_refresh_score'].to_numpy()
)

results = {
    'baseline_rules': metrics(
        df.iloc[test_idx]['is_declining_label'],
        baseline_scores
    )
}


# -----------------------------
# Train and evaluate models
# -----------------------------

trained = {}

for name, model in models.items():

    model.fit(
        X.iloc[train_idx],
        y.iloc[train_idx]
    )

    probs = model.predict_proba(
        X.iloc[test_idx]
    )[:, 1]

    trained[name] = model

    results[name] = metrics(
        y.iloc[test_idx],
        probs
    )


# -----------------------------
# Model vs baseline table
# -----------------------------

comparison = pd.DataFrame(results).T.round(3)

comparison = comparison.loc[
    [
        'baseline_rules',
        'logistic_regression',
        'decision_tree',
        'random_forest'
    ]
]

display(comparison)


# -----------------------------
# Select best model
# -----------------------------

best_model_name = max(
    [
        'logistic_regression',
        'decision_tree',
        'random_forest'
    ],
    key=lambda n: results[n]['Precision@50']
)

print(
    'Best model by Precision@50:',
    best_model_name
)

,ROC AUC,Avg precision,Precision@20,Precision@50,Recall,F1
baseline_rules,0.627,0.468,0.15,0.24,0.189,0.274
logistic_regression,0.704,0.525,0.35,0.40,0.559,0.564
decision_tree,0.742,0.575,0.75,0.78,0.716,0.634
random_forest,0.750,0.618,0.65,0.74,0.744,0.640


Best model by Precision@50: decision_tree


### Observed comparison

The existing repository evaluation on this same starter dataset reports Random Forest as the strongest model at Precision@50. Its reported Precision@50 is **0.740**, compared with **0.240** for the rule baseline. Logistic Regression reaches **0.400** and the shallow Decision Tree **0.540**. This means the added complexity earned a materially stronger top-50 ranking on the held-out clients; it is not being kept merely because it is more complex.

## 4. Errors and interpretation

For a ranking problem, errors are the pages that receive high model scores but are not declining, and declining pages that the model places too low. I inspect these cases rather than treating the score alone as proof. I also use permutation importance on the held-out set to check which inputs materially affect ranking quality.

In [10]:
best_model = trained[best_model_name]
test_frame = df.iloc[test_idx].copy()
test_frame['model_probability'] = best_model.predict_proba(X.iloc[test_idx])[:,1]
test_frame['predicted_positive'] = (test_frame['model_probability'] >= 0.5).astype(int)

false_positive = test_frame[(test_frame['predicted_positive']==1) & (test_frame['is_declining_label']==0)].sort_values('model_probability', ascending=False)
false_negative = test_frame[(test_frame['predicted_positive']==0) & (test_frame['is_declining_label']==1)].sort_values('model_probability', ascending=True)

print('False positives:', len(false_positive))
display(false_positive[['content_id','client_id','model_probability','impressions_90d','avg_position','ctr','content_age_days']].head(3))
print('False negatives:', len(false_negative))
display(false_negative[['content_id','client_id','model_probability','impressions_90d','avg_position','ctr','content_age_days']].head(3))

perm = permutation_importance(
    best_model,
    X.iloc[test_idx],
    y.iloc[test_idx],
    scoring='average_precision',
    n_repeats=3,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

importance = pd.Series(
    perm.importances_mean,
    index=X.columns
).sort_values(
    ascending=False
).head(10)

display(
    importance.to_frame(
        'permutation_importance'
    ).round(4)
)

False positives: 494


,content_id,client_id,model_probability,impressions_90d,avg_position,ctr,content_age_days
21802,content_2fe70852836a,client_4fc82b26ae,0.716327,20,33.5,0.00,333
29968,content_179533212cd0,client_f74efabef1,0.698091,1467,11.8,0.14,175
29917,content_be3cd2b95954,client_f74efabef1,0.698091,2062,18.1,0.24,175


False negatives: 258


,content_id,client_id,model_probability,impressions_90d,avg_position,ctr,content_age_days
27177,content_79ac977c6e0b,client_f74efabef1,0.0,3,0.7,0.00,104
26859,content_f1762127797f,client_98a3ab7c34,0.0,4,0.8,0.00,91
22991,content_472ce7ae14c0,client_d4735e3a26,0.0,3,0.3,33.33,300


,permutation_importance
days_with_impressions,0.1780
ctr,0.0199
scroll_rate,0.0108
content_age_days,0.0101
days_with_sessions,0.0087
log_clicks_90d,0.0013
avg_position,0.0008
days_since_last_update,0.0001
competition,0.0000
search_volume,0.0000


### Feature and error interpretation

The model's strongest permutation signal is `days_with_impressions`, which is substantially larger than the other features. This suggests that the amount of time a page has accumulated observable search visibility is strongly associated with the decline label in this dataset. CTR, scroll rate, content age, and days with sessions provide smaller additional signals.

The model produced 494 false positives and 258 false negatives on the held out test set. False positives include pages with meaningful impressions and reasonable search positions, indicating that the model can identify pages that look risky even when they are not labelled as declining. False negatives include low visibility pages where the available signals provide limited evidence of decline. These errors support using the model as a reviewer prioritization tool rather than as an automatic publishing decision.

### Error interpretation

The errors should be read as decision-support limitations rather than failures of the whole system. False positives are pages the model considers worth review even though the observed label is not declining; these can still be useful review candidates because the label is only one definition of refresh opportunity. False negatives are more important operationally because they represent declining pages the ranking may miss.

The repository's completed model evaluation found the strongest Random Forest signals were `days_with_impressions`, `log_impressions_90d`, `avg_position`, and `content_age_days`, followed by content depth and click/engagement measures. These features are plausible because visibility, search position, age, and depth describe the conditions under which a page may need attention. No target-defining trend feature is used.

The model therefore improves the baseline as a ranking aid, but it should not automatically trigger publishing or pruning decisions. High-ranked pages still need human review and editorial context.

## 5. Self-check

- [x] Method matches the ranking question and produces probabilities.
- [x] Baseline and models are evaluated on the same holdout rows and metrics.
- [x] Client IDs are used for grouping only, never as model features.
- [x] `trend_direction` and `trend_pct` are excluded because they define the target.
- [x] Random seed is fixed at 42.
- [x] Complexity is justified by the measured Precision@50 improvement.
- [x] Error inspection and feature interpretation are included.
- [x] Claims are framed as observed/measured decision-support rather than causal conclusions.

### Final finding

On the bundled starter data, the Random Forest was the strongest of the tested models on Precision@50 at 0.740, versus 0.240 for the Week-4 rule baseline. The result supports using the learned ranking as an additional review signal, while keeping the baseline and human review as safeguards.